# Práctica 1 — Módulo de riesgo: aplicación

**Objetivo.** Aplicar la librería `quantmgmt.risk` a cinco activos con más de 10 años de histórico diario: tabla resumen con todas las métricas, gráficos de drawdown, volatilidad y cola, y respuesta a las preguntas del enunciado.

**Activos.** Se eligen perfiles de riesgo distintos a propósito, para que las métricas discriminen:

| Ticker | Qué es | Por qué |
|---|---|---|
| SPY | Renta variable EE. UU. (S&P 500) | Referencia de mercado |
| TLT | Bonos del Tesoro EE. UU. +20 años | Activo defensivo, sensible a tipos |
| GLD | Oro | Diversificador, sin flujo de caja |
| HYG | Bonos corporativos high yield | Volatilidad baja pero riesgo de crédito en la cola |
| NVDA | Acción individual tecnológica | Rentabilidad y volatilidad extremas |

Además se construyen dos **carteras** con esos activos, para comparar NAVs de carteras con los de activos sueltos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from scipy.stats import norm

from quantmgmt.data import YahooFinanceData
from quantmgmt.risk import (
    drawdown_series,
    rolling_volatility,
    to_returns,
    var_historical,
    var_parametric,
    cvar_historical,
)
from quantmgmt.utils import generate_risk_summary, format_risk_summary

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

## 1. Datos

Precios **ajustados por splits y dividendos** (`Adj Close`), para medir rentabilidad total. La primera ejecución descarga de Yahoo Finance y guarda en `data/cache/`; las siguientes leen de disco.

In [ ]:
TICKERS = ["SPY", "TLT", "GLD", "HYG", "NVDA"]
START, END = "2014-01-01", "2025-12-31"

loader = YahooFinanceData()
assets = loader.get_prices(TICKERS, START, END)[TICKERS]

print(f"{assets.index[0]:%Y-%m-%d} -> {assets.index[-1]:%Y-%m-%d}: {len(assets)} sesiones")
assets.isna().sum()

**Tipo libre de riesgo.** Media del rendimiento del T-bill a 13 semanas (`^IRX`, en %) en el periodo. Se usa en Sharpe y Sortino: con `rf = 0` los ratios salen inflados, porque entre 2022 y 2025 los tipos superaron el 4 %.

In [ ]:
irx = loader.get_prices(["^IRX"], START, END)["^IRX"]
RF = irx.mean() / 100

print(f"rf medio {START[:4]}-{END[:4]}: {RF:.2%}")

**Carteras.** Dos construcciones sencillas con los mismos activos:

* **60/40**: 60 % SPY, 40 % TLT, rebalanceo diario (pesos constantes).
* **Equiponderada**: 20 % en cada activo, rebalanceo diario.

El NAV de cada cartera se construye a partir de los retornos diarios ponderados y empieza en 100, así que la librería lo trata igual que el precio de un activo.

In [ ]:
def constant_weight_nav(prices: pd.DataFrame, weights: dict[str, float]) -> pd.Series:
    """NAV (base 100) de una cartera con pesos constantes, rebalanceada cada periodo."""
    w = pd.Series(weights)
    components = prices[w.index].dropna()
    portfolio_returns = to_returns(components).dropna().dot(w)
    nav = 100 * (1 + portfolio_returns).cumprod()
    # El NAV empieza en 100 el primer día con precio de todos los componentes
    return pd.concat([pd.Series(100.0, index=components.index[:1]), nav])


portfolios = pd.DataFrame({
    "60/40": constant_weight_nav(assets, {"SPY": 0.6, "TLT": 0.4}),
    "Equiponderada": constant_weight_nav(assets, {t: 1 / len(TICKERS) for t in TICKERS}),
})

prices = assets.join(portfolios)

(prices / prices.bfill().iloc[0] * 100).plot(logy=True, title="Evolución (base 100, escala logarítmica)")
plt.show()

## 2. Tabla resumen

Todas las métricas del módulo, más la métrica propia. Convenciones:

* Rentabilidades, volatilidades y tracking error, **anualizados**.
* VaR y CVaR, **diarios** y como pérdida positiva.
* Tiempos (`Max Time Under Water`, `Recovery Time`) en **sesiones**.
* Tracking error frente a SPY.

In [ ]:
summary = generate_risk_summary(prices, risk_free_rate=RF, benchmark_prices=prices["SPY"])

format_risk_summary(summary, transpose=True)

## 3. Gráficos

### 3.1 Drawdown

Caída respecto al máximo alcanzado hasta cada fecha (*underwater plot*). Muestra a la vez la profundidad y la duración de las pérdidas.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
drawdown_series(prices).plot(ax=ax, lw=1)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title("Drawdown respecto al máximo previo")
plt.show()

### 3.2 Volatilidad rolling (63 sesiones ≈ 3 meses, anualizada)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
rolling_volatility(prices, window=63).plot(ax=ax, lw=1)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title("Volatilidad anualizada, ventana de 63 sesiones")
plt.show()

### 3.3 Distribución de retornos diarios con VaR y CVaR al 95 %

Líneas: VaR histórico (naranja), CVaR histórico (rojo) y VaR gaussiano (azul discontinuo). Si el CVaR queda lejos del VaR, la cola es profunda: cuando se pierde más que el VaR, se pierde mucho más. El eje x se recorta al 99,9 % de los datos para que se vea la forma.

In [ ]:
returns = to_returns(prices)
var95 = var_historical(prices, 0.95)
cvar95 = cvar_historical(prices, 0.95)
var95_gauss = var_parametric(prices, 0.95)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, name in zip(axes.flat, prices.columns):
    r = returns[name].dropna()
    ax.hist(r, bins=120, color="lightgray")
    ax.axvline(-var95[name], color="tab:orange", lw=2, label=f"VaR hist {var95[name]:.2%}")
    ax.axvline(-cvar95[name], color="tab:red", lw=2, label=f"CVaR hist {cvar95[name]:.2%}")
    ax.axvline(-var95_gauss[name], color="tab:blue", ls="--", label=f"VaR gauss {var95_gauss[name]:.2%}")
    lim = r.abs().quantile(0.999)
    ax.set_xlim(-lim, lim)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.set_title(name)
    ax.legend(fontsize=8)

for ax in axes.flat[len(prices.columns):]:
    ax.axis("off")

fig.suptitle("Retornos diarios con VaR y CVaR al 95 %")
fig.tight_layout()
plt.show()

## 4. Preguntas

### 4.1 Nos dan 1 M€. ¿Nos convence alguna cartera? ¿Cuál y por qué?

Traducimos las métricas a euros sobre 1 M€ y ordenamos por *Tail-Adjusted Sharpe*, que penaliza tanto la volatilidad como la cola.

In [ ]:
CAPITAL = 1_000_000

in_euros = pd.DataFrame({
    "CAGR": summary["CAGR"].map("{:.1%}".format),
    "Peor caída (€)": (summary["Maximum Drawdown"] * CAPITAL).map("{:,.0f}".format),
    "Pérdida media en el 5 % peor de días (€)": (-summary["CVaR 95% Historical"] * CAPITAL).map("{:,.0f}".format),
    "Sesiones sin recuperar máximo": summary["Max Time Under Water"],
    "Sharpe": summary["Sharpe Ratio"].round(2),
    "Tail-Adj. Sharpe": summary["Tail-Adjusted Sharpe"].round(2),
    "Calmar": summary["Calmar Ratio"].round(2),
}).loc[summary["Tail-Adjusted Sharpe"].sort_values(ascending=False).index]

in_euros

**Respuesta:**

_(Completar tras ejecutar. Qué mirar: qué opción combina mejor rentabilidad y riesgo ajustado, si la peor caída en euros sería tolerable para quien pone el dinero, y cuánto tardaría en recuperarse. Comparar NVDA frente a las carteras: la rentabilidad pasada de un solo valor no es una estrategia replicable.)_

### 4.2 ¿Qué activo tiene el peor perfil de cola y por qué no se aprecia mirando solo la volatilidad?

La volatilidad supone implícitamente una cola normal. Comparamos el VaR al 99 % que predice la normal con el observado, junto con la asimetría, la curtosis y el *Tail Excess Ratio* (τ) de la métrica propia.

In [ ]:
tail_profile = pd.DataFrame({
    "Volatilidad": summary["Annualized Volatility"],
    "Asimetría": summary["Skewness"],
    "Exceso curtosis": summary["Excess Kurtosis"],
    "VaR 99% gaussiano": var_parametric(prices, 0.99),
    "VaR 99% histórico": var_historical(prices, 0.99),
    "VaR 99% Cornish-Fisher": var_parametric(prices, 0.99, distribution="cornish_fisher"),
    "Tail Excess Ratio (τ)": summary["Tail Excess Ratio"],
})
tail_profile["Histórico / gaussiano"] = tail_profile["VaR 99% histórico"] / tail_profile["VaR 99% gaussiano"]

tail_profile.sort_values("Tail Excess Ratio (τ)", ascending=False).style.format({
    "Volatilidad": "{:.1%}", "VaR 99% gaussiano": "{:.2%}", "VaR 99% histórico": "{:.2%}",
    "VaR 99% Cornish-Fisher": "{:.2%}", "Asimetría": "{:.2f}", "Exceso curtosis": "{:.1f}",
    "Tail Excess Ratio (τ)": "{:.2f}", "Histórico / gaussiano": "{:.2f}",
})

In [ ]:
rank_compare = pd.DataFrame({
    "Rank por volatilidad": summary["Annualized Volatility"].rank(ascending=False),
    "Rank por τ (forma de la cola)": summary["Tail Excess Ratio"].rank(ascending=False),
}).astype(int).sort_values("Rank por τ (forma de la cola)")

rank_compare

**Respuesta:**

_(Completar. Qué mirar: el activo con mayor τ, exceso de curtosis y cociente histórico/gaussiano; si está abajo en volatilidad y arriba en τ, esa es la respuesta a "por qué no se aprecia con la volatilidad". HYG es el candidato esperado: pocas oscilaciones en condiciones normales y caídas bruscas en crisis de crédito, como marzo de 2020.)_

### 4.3 ¿Coincide el ranking por Sharpe con el ranking por Calmar?

Sharpe mide el riesgo con toda la dispersión; Calmar, con el peor episodio. Se comparan los rankings y su correlación de Spearman (1 = mismo orden).

In [ ]:
rankings = pd.DataFrame({
    "Sharpe": summary["Sharpe Ratio"],
    "Rank Sharpe": summary["Sharpe Ratio"].rank(ascending=False).astype(int),
    "Calmar": summary["Calmar Ratio"],
    "Rank Calmar": summary["Calmar Ratio"].rank(ascending=False).astype(int),
    "Max drawdown": summary["Maximum Drawdown"],
}).sort_values("Rank Sharpe")

spearman = summary["Sharpe Ratio"].corr(summary["Calmar Ratio"], method="spearman")
print(f"Correlación de Spearman entre rankings: {spearman:.2f}")

rankings.style.format({"Sharpe": "{:.2f}", "Calmar": "{:.2f}", "Max drawdown": "{:.1%}"})

**Respuesta:**

_(Completar. Qué mirar: qué activos cambian de posición y por qué. Un activo con volatilidad moderada pero un único drawdown profundo, como TLT en 2022, sube en Sharpe y baja en Calmar. Calmar depende de un solo episodio, así que es más inestable ante la ventana elegida.)_

### 4.4 ¿Cómo se comporta el VaR paramétrico en dos periodos diferentes?

El VaR gaussiano depende solo de la media y la volatilidad de la ventana en la que se estima. Se compara:

* **Periodo tranquilo:** estimado con 2019 y aplicado a 2020 (COVID).
* **Periodo estresado:** estimado con 2020 y aplicado a 2021.

Si el VaR al 95 % fuera correcto, se superaría en torno al **5 %** de los días del año siguiente.

In [ ]:
def var_backtest(prices, estimation, test, confidence=0.95):
    """VaR gaussiano estimado en una ventana y % de días que lo superan en la siguiente."""
    var = var_parametric(prices.loc[estimation[0]:estimation[1]], confidence)
    test_returns = to_returns(prices.loc[test[0]:test[1]]).dropna(how="all")
    exceedances = test_returns.lt(-var).sum() / test_returns.count()
    return var, exceedances


var_2019, exc_2020 = var_backtest(assets, ("2019-01-01", "2019-12-31"), ("2020-01-01", "2020-12-31"))
var_2020, exc_2021 = var_backtest(assets, ("2020-01-01", "2020-12-31"), ("2021-01-01", "2021-12-31"))

pd.DataFrame({
    "VaR 95% estimado en 2019": var_2019,
    "% días que lo superan en 2020": exc_2020,
    "VaR 95% estimado en 2020": var_2020,
    "% días que lo superan en 2021": exc_2021,
}).style.format("{:.2%}")

Visualmente, para SPY: VaR gaussiano rolling (ventana de 1 año) frente a los retornos del día siguiente. Los puntos rojos son los días que lo superan.

In [ ]:
spy = returns["SPY"].dropna()
window = 252
z = norm.ppf(0.05)
rolling_var = -(spy.rolling(window).mean() + z * spy.rolling(window).std()).shift(1)
breaches = spy < -rolling_var

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(spy.index, spy, color="lightgray", lw=0.6, label="Retorno diario")
ax.plot(rolling_var.index, -rolling_var, color="tab:blue", lw=1.5, label="-VaR 95% gaussiano (1 año)")
ax.scatter(spy.index[breaches], spy[breaches], color="tab:red", s=8, label="Días que lo superan")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title(f"SPY: VaR gaussiano rolling. Superado el {breaches[rolling_var.notna()].mean():.1%} de los días (objetivo 5 %)")
ax.legend()
plt.show()

**Respuesta:**

_(Completar. Qué mirar: estimado en un año tranquilo, el VaR es bajo y en el año siguiente de crisis se supera muchas más veces del 5 %. Estimado en un año de crisis, es alto y en el año siguiente se supera menos del 5 %. El VaR paramétrico reacciona con retraso: infravalora el riesgo justo antes de las crisis y lo sobrevalora después.)_

### 4.5 ¿Qué nos dice la métrica propia?

**Volatilidad ajustada por cola (TAV):** $\text{TAV} = \sigma \cdot \max(1, \tau)$, con $\tau$ = CVaR histórico / CVaR gaussiano (al 97,5 %, sobre retornos centrados). Documentación completa en `quantmgmt/risk/custom.py`.

Si τ ≈ 1, la volatilidad cuenta toda la historia. Si τ > 1, hay riesgo de cola que la volatilidad esconde. El *Tail-Adjusted Sharpe* sustituye σ por la TAV en el denominador.

In [ ]:
own_metric = pd.DataFrame({
    "Volatilidad": summary["Annualized Volatility"],
    "τ": summary["Tail Excess Ratio"],
    "TAV": summary["Tail-Adjusted Volatility"],
    "Penalización": summary["Tail-Adjusted Volatility"] / summary["Annualized Volatility"] - 1,
    "Sharpe": summary["Sharpe Ratio"],
    "Tail-Adj. Sharpe": summary["Tail-Adjusted Sharpe"],
    "Rank Sharpe": summary["Sharpe Ratio"].rank(ascending=False).astype(int),
    "Rank Tail-Adj. Sharpe": summary["Tail-Adjusted Sharpe"].rank(ascending=False).astype(int),
}).sort_values("τ", ascending=False)

own_metric.style.format({
    "Volatilidad": "{:.1%}", "τ": "{:.2f}", "TAV": "{:.1%}", "Penalización": "{:+.1%}",
    "Sharpe": "{:.2f}", "Tail-Adj. Sharpe": "{:.2f}",
})

**Respuesta:**

_(Completar. Qué mirar: qué activos reciben más penalización, si coincide con el peor perfil de cola de 4.2, y si algún activo cambia de posición al pasar de Sharpe a Tail-Adjusted Sharpe. Si nadie cambia de posición, también es un resultado: el ranking por Sharpe es robusto para estos activos.)_

## 5. Conclusiones

_(Completar con 3-4 ideas principales.)_